# 02. Статистические модели прогнозирования

Во втором ноутбуке выполнено сравнение статистических методов прогнозирования временного ряда энергопотребления: простые baseline, ручные модели, автоматические модели, интервальные прогнозы, rolling backtesting и анализ остатков.

## Связь с заданием

Раздел закрывает второй пункт задания: используется `statsforecast`, сравниваются более 5 методов, включены ARIMA/ETS/Theta в ручном и автоматическом вариантах, выполнен backtesting, построены интервальные прогнозы и проанализированы остатки. Prophet добавлен как дополнительный ориентир вне `statsforecast`.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsforecast import StatsForecast
from statsforecast.models import (
    ARIMA,
    AutoARIMA,
    AutoETS,
    AutoTheta,
    HoltWinters,
    HistoricAverage,
    Naive,
    SeasonalNaive,
    Theta,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## Конфигурация запуска

В финальной версии используется режим полного сравнения: `QUICK_MODE=False`. В отдельном статистическом ноутбуке автоматические модели находятся в конфигурации запуска, включая AutoARIMA, AutoETS и AutoTheta. Это делает ноутбук более тяжелым по времени выполнения, чем объединенный финальный отчет.

In [9]:
QUICK_MODE = False

forecast_horizon = 24
test_start = "2019-01-01"
prediction_levels = [80, 95]

if QUICK_MODE:
    train_history_days = 365
    n_windows = 2
    include_auto_models = False
    n_jobs = 1
else:
    train_history_days = None
    n_windows = 8
    include_auto_models = True
    n_jobs = -1

print("QUICK_MODE:", QUICK_MODE)
print("Горизонт прогноза, часов:", forecast_horizon)
print("Окна backtesting:", n_windows)
print("Длина рабочей train-истории, дней:", train_history_days)
print("Автоматические модели включены:", include_auto_models)

QUICK_MODE: False
Горизонт прогноза, часов: 24
Окна backtesting: 8
Длина рабочей train-истории, дней: None
Автоматические модели включены: True


## Загрузка данных

Основной входной файл для раздела - подготовленный датасет из `data/processed`. В коде также оставлен fallback на исходный CSV из `data/raw`, чтобы раздел оставался воспроизводимым при отсутствии prepared-файла.

In [10]:
processed_path = Path("../data/processed/de_hourly_power_and_weather_prepared.csv")
raw_path = Path("../data/raw/de_hourly_power_and_weather.csv")

if processed_path.exists():
    df = pd.read_csv(processed_path, parse_dates=["timestamp"])
    df = df.rename(columns={"timestamp": "ds"})
else:
    df = pd.read_csv(raw_path, parse_dates=["utc_timestamp"])
    df = df.rename(columns={"utc_timestamp": "ds"})

df["ds"] = pd.to_datetime(df["ds"], utc=True).dt.tz_convert(None)
df = df.sort_values("ds").reset_index(drop=True)

df.head()

,ds,Consumption,Wind,Solar,Wind+Solar,temperature,radiation_direct_horizontal,radiation_diffuse_horizontal
0,2015-01-01 00:00:00,41.151,8.852,NaN,NaN,-0.981,0.0,0.0
1,2015-01-01 01:00:00,40.135,9.054,NaN,NaN,-1.035,0.0,0.0
2,2015-01-01 02:00:00,39.106,9.070,NaN,NaN,-1.109,0.0,0.0
3,2015-01-01 03:00:00,38.765,9.163,NaN,NaN,-1.166,0.0,0.0
4,2015-01-01 04:00:00,38.941,9.231,NaN,NaN,-1.226,0.0,0.0


## Формат данных для `statsforecast`

`statsforecast` использует длинный формат `unique_id`, `ds`, `y`. У нас один временной ряд, поэтому `unique_id` фиксирован.

In [11]:
target_col = "Consumption"

sf_df = df[["ds", target_col]].rename(columns={target_col: "y"}).copy()
sf_df.insert(0, "unique_id", "DE_consumption")

train_full_sf = sf_df.loc[sf_df["ds"] < test_start].copy()
test_sf = sf_df.loc[sf_df["ds"] >= test_start].copy()

if train_history_days is None:
    train_sf = train_full_sf.copy()
else:
    train_start = pd.Timestamp(test_start) - pd.Timedelta(days=train_history_days)
    train_sf = train_full_sf.loc[train_full_sf["ds"] >= train_start].copy()

print("Полный ряд:", sf_df["ds"].min(), "->", sf_df["ds"].max(), sf_df.shape)
print("Полный train:", train_full_sf["ds"].min(), "->", train_full_sf["ds"].max(), train_full_sf.shape)
print("Рабочий train:", train_sf["ds"].min(), "->", train_sf["ds"].max(), train_sf.shape)
print("Test:", test_sf["ds"].min(), "->", test_sf["ds"].max(), test_sf.shape)

sf_df.head()

Полный ряд: 2015-01-01 00:00:00 -> 2019-12-31 23:00:00 (43824, 3)
Полный train: 2015-01-01 00:00:00 -> 2018-12-31 23:00:00 (35064, 3)
Рабочий train: 2015-01-01 00:00:00 -> 2018-12-31 23:00:00 (35064, 3)
Test: 2019-01-01 00:00:00 -> 2019-12-31 23:00:00 (8760, 3)


,unique_id,ds,y
0,DE_consumption,2015-01-01 00:00:00,41.151
1,DE_consumption,2015-01-01 01:00:00,40.135
2,DE_consumption,2015-01-01 02:00:00,39.106
3,DE_consumption,2015-01-01 03:00:00,38.765
4,DE_consumption,2015-01-01 04:00:00,38.941


## Метрики качества

Используем MAE, RMSE и SMAPE. SMAPE выбрана вместо MAPE как более симметричная относительная метрика.

In [12]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    errors = np.where(denominator == 0, 0, np.abs(y_true - y_pred) / denominator)
    return np.mean(errors) * 100


def forecast_columns(columns):
    ignored = {"unique_id", "ds", "cutoff", "y"}
    return [
        col for col in columns
        if col not in ignored and "-lo-" not in col and "-hi-" not in col
    ]


def evaluate_forecasts(forecasts):
    rows = []
    y_true = forecasts["y"].to_numpy()

    for model in forecast_columns(forecasts.columns):
        y_pred = forecasts[model].to_numpy()
        rows.append({
            "model": model,
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "SMAPE": smape(y_true, y_pred),
        })

    return pd.DataFrame(rows).sort_values("SMAPE").reset_index(drop=True)

## Набор моделей

Бейзлайны задают нижнюю планку качества. Ручные ARIMA/ETS/Theta показывают контролируемые настройки. Автоматические AutoARIMA/AutoETS/AutoTheta проверяют подбор параметров средствами статистических моделей.

В этом отдельном ноутбуке AutoARIMA сохранена в списке моделей. Ее высокая вычислительная стоимость зафиксирована как практическое ограничение: на длинном почасовом ряде полный rolling backtesting с AutoARIMA выполняется существенно дольше остальных моделей.

In [13]:
baseline_models = [
    HistoricAverage(alias="HistoricAverage"),
    Naive(alias="Naive"),
    SeasonalNaive(season_length=24, alias="SeasonalNaive_24h"),
    SeasonalNaive(season_length=168, alias="SeasonalNaive_168h"),
]

manual_models = [
    ARIMA(order=(1, 1, 1), season_length=24, alias="ARIMA_111_24h"),
    HoltWinters(season_length=24, error_type="A", alias="HoltWinters_AAA_24h"),
    Theta(season_length=24, alias="Theta_24h"),
]

auto_models = [
    AutoARIMA(season_length=24, alias="AutoARIMA_24h"),
    AutoETS(season_length=24, alias="AutoETS_24h"),
    AutoTheta(season_length=24, alias="AutoTheta_24h"),
]

models = baseline_models + manual_models
if include_auto_models:
    models += auto_models

sf = StatsForecast(models=models, freq="h", n_jobs=n_jobs)
models

[HistoricAverage,
 Naive,
 SeasonalNaive_24h,
 SeasonalNaive_168h,
 ARIMA_111_24h,
 HoltWinters_AAA_24h,
 Theta_24h,
 AutoARIMA_24h,
 AutoETS_24h,
 AutoTheta_24h]

## Rolling backtesting

Каждое окно прогнозирует следующие 24 часа, затем окно сдвигается на 24 часа. В backtesting дополнительно рассчитаны интервалы 80% и 95%, чтобы оценить не только точечный, но и интервальный прогноз.

In [14]:
cv_df = sf.cross_validation(
    df=train_sf,
    h=forecast_horizon,
    step_size=forecast_horizon,
    n_windows=n_windows,
    level=prediction_levels,
)

cv_df.head()

KeyboardInterrupt: 

In [ ]:
cv_scores = evaluate_forecasts(cv_df)
cv_scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_scores.sort_values("SMAPE").plot.barh(x="model", y="SMAPE", ax=ax, legend=False)
ax.set_title("Сравнение моделей по SMAPE на rolling backtesting")
ax.set_xlabel("SMAPE, %")
ax.set_ylabel("Модель")
plt.tight_layout()

## Финальный прогноз на тестовом периоде

После backtesting модели обучаются на рабочем train-периоде и строят прогноз на первые 24 часа test-периода. Этот блок показывает, как выбранные статистические модели ведут себя на фиксированном финальном горизонте.

In [ ]:
sf.fit(train_sf)
forecast_df = sf.predict(h=forecast_horizon, level=prediction_levels)

test_24h = test_sf.head(forecast_horizon)
final_eval_df = test_24h.merge(forecast_df, on=["unique_id", "ds"], how="left")

final_eval_df.head()

In [ ]:
final_scores = evaluate_forecasts(final_eval_df)
final_scores

In [ ]:
best_model = cv_scores.iloc[0]["model"]
history_plot = train_sf.tail(24 * 7)
lo_col = f"{best_model}-lo-80"
hi_col = f"{best_model}-hi-80"

def to_mpl_dates(values):
    return mdates.date2num(pd.to_datetime(values).to_numpy(dtype="datetime64[ms]"))

history_x = to_mpl_dates(history_plot["ds"])
test_x = to_mpl_dates(test_24h["ds"])
forecast_x = to_mpl_dates(final_eval_df["ds"])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(history_x, history_plot["y"].to_numpy(), label="История, последние 7 дней train", linewidth=0.8)
ax.plot(test_x, test_24h["y"].to_numpy(), label="Факт, первые 24 часа test", linewidth=1.2)
ax.plot(forecast_x, final_eval_df[best_model].to_numpy(), label=f"Прогноз: {best_model}", linewidth=1.2)

if lo_col in final_eval_df.columns and hi_col in final_eval_df.columns:
    ax.fill_between(
        forecast_x,
        final_eval_df[lo_col].astype(float).to_numpy(),
        final_eval_df[hi_col].astype(float).to_numpy(),
        alpha=0.2,
        label="Интервал 80%",
    )

ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d\n%H:%M"))
ax.set_title("Финальный 24-часовой прогноз лучшей модели")
ax.set_xlabel("Время")
ax.set_ylabel("Энергопотребление")
ax.legend()
plt.tight_layout()

## Prophet

Prophet добавлен как отдельный сравнительный блок вне `statsforecast`. Он используется как дополнительный ориентир качества для первого 24-часового прогноза test-периода.

In [ ]:
from prophet import Prophet

prophet_train = train_sf[["ds", "y"]].copy()
prophet_model = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
prophet_model.fit(prophet_train)

prophet_future = test_24h[["ds"]].copy()
prophet_forecast = prophet_model.predict(prophet_future)[["ds", "yhat", "yhat_lower", "yhat_upper"]]
prophet_eval = test_24h[["ds", "y"]].merge(prophet_forecast, on="ds", how="left")

prophet_result = pd.DataFrame([
    {
        "model": "Prophet",
        "MAE": mae(prophet_eval["y"].to_numpy(), prophet_eval["yhat"].to_numpy()),
        "RMSE": rmse(prophet_eval["y"].to_numpy(), prophet_eval["yhat"].to_numpy()),
        "SMAPE": smape(prophet_eval["y"].to_numpy(), prophet_eval["yhat"].to_numpy()),
    }
])

final_scores_with_prophet = (
    pd.concat([final_scores, prophet_result], ignore_index=True)
    .sort_values("SMAPE")
    .reset_index(drop=True)
)

final_scores_with_prophet

## Анализ остатков лучшей модели

Остатки показывают, где модель систематически ошибается. Для качественной модели остатки похожи на шум и не имеют выраженного временного паттерна.

In [ ]:
residuals_df = final_eval_df[["ds", "y", best_model]].copy()
residuals_df["residual"] = residuals_df["y"] - residuals_df[best_model]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

residuals_df.plot(x="ds", y="residual", ax=axes[0], marker="o", legend=False)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Остатки во времени")
axes[0].set_xlabel("Время")
axes[0].set_ylabel("Факт - прогноз")

residuals_df["residual"].plot.hist(ax=axes[1], bins=12)
axes[1].set_title("Распределение остатков")
axes[1].set_xlabel("Остаток")

plt.tight_layout()

## Обоснование выбора методов

- `Naive`, `SeasonalNaive`, `HistoricAverage` задают простые нижние ориентиры качества.
- `ARIMA_111_24h` проверяет ручной ARIMA-подход с дифференцированием и суточной сезонностью.
- `HoltWinters_AAA_24h` проверяет ручную ETS/Holt-Winters модель с аддитивной ошибкой, трендом и сезонностью.
- `Theta_24h` проверяет классический Theta-подход для сезонного ряда.
- `AutoARIMA`, `AutoETS` и `AutoTheta` проверяют автоматический подбор параметров.

Такой набор покрывает простые baseline, классические статистические модели и автоматический подбор параметров.

## Итоги

In [ ]:
summary = {
    "quick_mode": QUICK_MODE,
    "forecast_horizon_hours": forecast_horizon,
    "prediction_levels": prediction_levels,
    "n_backtesting_windows": n_windows,
    "models_compared": forecast_columns(cv_df.columns),
    "best_model_by_cv_smape": best_model,
    "cv_scores": cv_scores.to_dict(orient="records"),
    "final_24h_scores": final_scores_with_prophet.to_dict(orient="records"),
}

stat_config_df = pd.DataFrame([
    {"Показатель": "QUICK_MODE", "Значение": QUICK_MODE},
    {"Показатель": "Горизонт прогноза, часов", "Значение": forecast_horizon},
    {"Показатель": "Уровни интервалов", "Значение": ", ".join(map(str, prediction_levels))},
    {"Показатель": "Окна backtesting", "Значение": n_windows},
    {"Показатель": "Дополнительная модель", "Значение": "Prophet"},
])

best_model_df = pd.DataFrame([
    {
        "Критерий": "Минимальный SMAPE на rolling backtesting",
        "Лучшая модель": best_model,
        "SMAPE": cv_scores.iloc[0]["SMAPE"],
    }
])

models_compared_df = pd.DataFrame({"Модель": summary["models_compared"]})

print("Конфигурация статистического сравнения")
display(stat_config_df)
print("Лучшая модель по rolling backtesting")
display(best_model_df)
print("Метрики rolling backtesting")
display(cv_scores)
print("Метрики финального 24-часового прогноза")
display(final_scores_with_prophet)
print("Модели в сравнении")
display(models_compared_df)

Финальное сравнение выполнено в режиме `QUICK_MODE=False`. Для AutoARIMA в работе зафиксировано вычислительное ограничение: модель формально включена в отдельный статистический ноутбук, но именно она делает полный rolling backtesting наиболее затратным по времени.

## Интерпретация результатов

По сохраненной таблице rolling backtesting лучшей статистической моделью стала `SeasonalNaive_168h` со `SMAPE` около 2.68%. Это сильный результат для простой базовой модели и прямое подтверждение важности недельной сезонности в ряде энергопотребления.

`SeasonalNaive_24h` оказался слабее недельной наивной модели, поэтому одного суточного паттерна недостаточно: профиль потребления зависит не только от часа суток, но и от дня недели. `HoltWinters_AAA_24h` оказался близок к суточной сезонной модели, но не превзошел недельный baseline. `Theta`, `Naive`, `HistoricAverage` и ручная `ARIMA_111_24h` дали более слабое качество.

Главный вывод статистического раздела: простые сезонные модели являются сильной точкой сравнения. Более сложные статистические модели имеют смысл только тогда, когда они улучшают `SeasonalNaive_24h` или `SeasonalNaive_168h` по `SMAPE`.